# DistilBERT Text Classification Model

This notebook trains a DistilBERT model on the preprocessed text data with an 80-10-10 train-validation-test split.

## Section 1: Load and Prepare Data

Load the preprocessed data and verify that the 'other_posts' and 'label' columns are ready for model training.

In [1]:
import subprocess
import sys
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [2]:


# Load the preprocessed data
data = pd.read_csv('second_dataset.csv')
data= data.sample(frac=1, random_state=42).reset_index(drop=True)

# Display basic information about the data
print("Dataset shape:", data.shape)
print("\nFirst few rows:")
print(data.head())
print("\nColumn names:")
print(data.columns.tolist())
print("\nData types:")
print(data.dtypes)
print("\nLabel distribution:")
print(data['label'].value_counts())

Dataset shape: (98383, 11)

First few rows:
              author      subreddit  \
0       HarrisonM115   mentalhealth   
1          Heberlein  relationships   
2        Whoretortle        divorce   
3  Aromatic_Honeydew  relationships   
4          biancaluz  relationships   

                                         report_post  \
0  It's like i have this version of myself that j...   
1                                                NaN   
2                                                NaN   
3                                                NaN   
4                                                NaN   

                                         other_posts  is_self_report  \
0  it's like i have this version of myself that j...               1   
1  I [23M] messed things up with my date [23F] on...               0   
2  Blood test for alcohol, divorce, ex-wife. My e...               0   
3  I (25F) lost sexual interest in my boyfriend (...               0   
4  This is the 3rd time 

In [3]:
def minimal_preprocess(text):
    text = text.lower()
    #  Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove usernames (Twitter/Reddit style)
    text = re.sub(r'@\w+', '', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


In [4]:

data['other_posts']= data['other_posts'].apply(minimal_preprocess)

## Section 2: Split Data into Train, Validation, and Test Sets

Use scikit-learn's train_test_split to create 80% training, 10% validation, and 10% test sets.

In [5]:
# Split data into train (80%) and temp (20%)
train_data, temp_data = train_test_split(
    data, 
    test_size=0.2, 
    random_state=42, 
    stratify=data['label']
)

# Split temp (20%) into validation (50% of 20% = 10%) and test (50% of 20% = 10%)
val_data, test_data = train_test_split(
    temp_data, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_data['label']
)

print(f"Training set size: {len(train_data)} ({len(train_data)/len(data)*100:.1f}%)")
print(f"Validation set size: {len(val_data)} ({len(val_data)/len(data)*100:.1f}%)")
print(f"Test set size: {len(test_data)} ({len(test_data)/len(data)*100:.1f}%)")

print("\nLabel distribution in each set:")
print("Training set:")
print(train_data['label'].value_counts())
print("\nValidation set:")
print(val_data['label'].value_counts())
print("\nTest set:")
print(test_data['label'].value_counts())

# Reset indices
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

Training set size: 78706 (80.0%)
Validation set size: 9838 (10.0%)
Test set size: 9839 (10.0%)

Label distribution in each set:
Training set:
label
0    48000
1    30706
Name: count, dtype: int64

Validation set:
label
0    6000
1    3838
Name: count, dtype: int64

Test set:
label
0    6000
1    3839
Name: count, dtype: int64


## Section 3: Tokenize Text with DistilBERT

Use DistilBertTokenizer to tokenize the 'other_posts' text data with appropriate padding and truncation.

In [6]:
# Initialize the DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define tokenization function
def tokenize_function(texts, max_length=512):
    """
    Tokenize texts using DistilBERT tokenizer
    """
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encodings


In [9]:
# Tokenize train, validation, and test sets
print("Tokenizing training data...")
train_encodings = tokenize_function(train_data['other_posts'])

print("Tokenizing validation data...")
val_encodings = tokenize_function(val_data['other_posts'])

print("Tokenizing test data...")
test_encodings = tokenize_function(test_data['other_posts'])

print("\nTokenization complete!")
print(f"Train encodings shape: input_ids={train_encodings['input_ids'].shape}")
print(f"Validation encodings shape: input_ids={val_encodings['input_ids'].shape}")
print(f"Test encodings shape: input_ids={test_encodings['input_ids'].shape}")

Tokenizing training data...
Tokenizing validation data...
Tokenizing test data...

Tokenization complete!
Train encodings shape: input_ids=torch.Size([78706, 512])
Validation encodings shape: input_ids=torch.Size([9838, 512])
Test encodings shape: input_ids=torch.Size([9839, 512])


## Section 4: Create PyTorch Datasets

Create custom PyTorch Dataset classes for train, validation, and test sets with tokenized inputs and labels.

In [10]:
class TextClassificationDataset(Dataset):
    """
    Custom PyTorch Dataset for text classification with tokenized inputs
    """
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


In [11]:

# Create datasets
train_dataset = TextClassificationDataset(train_encodings, train_data['label'])
val_dataset = TextClassificationDataset(val_encodings, val_data['label'])
test_dataset = TextClassificationDataset(test_encodings, test_data['label'])

print("Dataset Creation Complete!")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nData loaders created with batch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Dataset Creation Complete!
Train dataset size: 78706
Validation dataset size: 9838
Test dataset size: 9839

Data loaders created with batch size: 16
Training batches: 4920
Validation batches: 615
Test batches: 615


## Section 5: Initialize DistilBERT Model

Load a pretrained DistilBERT model for sequence classification and configure training parameters.

In [12]:
# Determine number of labels
num_labels = len(data['label'].unique())
print(f"Number of unique labels: {num_labels}")

# Load pretrained DistilBERT model for sequence classification
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Number of unique labels: 2
Using device: cuda


In [13]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

# Move model to device
model.to(device)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [14]:

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

# Move model to device
model.to(device)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [15]:
def seed_everything():
    np.random(42)
    torch.random(42)
    return

In [16]:

# Define optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * 3  # 3 epochs
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, total_iters=total_steps)

print("\nModel Configuration:")
print(f"Model: DistilBERT for Sequence Classification")
print(f"Number of labels: {num_labels}")
print(f"Total training steps: {total_steps}")
print(f"Learning rate: 2e-5")
print(f"Device: {device}")


Model Configuration:
Model: DistilBERT for Sequence Classification
Number of labels: 2
Total training steps: 14760
Learning rate: 2e-5
Device: cuda


## Section 6: Train the Model

Train the DistilBERT model on the training set using a training loop, with validation on the validation set after each epoch.

In [17]:
def train_epoch(model, train_loader, optimizer, scheduler, device):
    """
    Train the model for one epoch
    """
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc="Training"):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss

def evaluate(model, eval_loader, device):
    """
    Evaluate the model on validation or test set
    """
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating"):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            
            # Forward pass
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item()
            
            # Get predictions
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(eval_loader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy, predictions, true_labels


In [18]:

# Training parameters
num_epochs = 3
best_val_accuracy = 0
patience = 1
patience_counter = 0

print("Starting model training...\n")

# Training loop
for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"Training Loss: {train_loss:.4f}")
    
    # Validate
    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, device)
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    
    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), 'best_distilbert_model_v2.pt')
        print("Best model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

print("\nTraining completed!")

Starting model training...


Epoch 1/3


Training: 100%|██████████| 4920/4920 [37:36<00:00,  2.18it/s]


Training Loss: 0.1757


Evaluating: 100%|██████████| 615/615 [01:21<00:00,  7.57it/s]


Validation Loss: 0.1185
Validation Accuracy: 0.9532
Best model saved!

Epoch 2/3


...

Validation Loss: 0.1150
Validation Accuracy: 0.9588
Best model saved!

Epoch 3/3


Training: 100%|██████████| 4920/4920 [37:49<00:00,  2.17it/s]


Training Loss: 0.0678


Evaluating: 100%|██████████| 615/615 [01:21<00:00,  7.55it/s]

Validation Loss: 0.1234
Validation Accuracy: 0.9569
Early stopping triggered after 3 epochs

Training completed!


In [19]:

# print("Starting model training...\n")

# # Training loop
# for epoch in range(num_epochs):
#     print(f"\n{'='*50}")
#     print(f"Epoch {epoch + 1}/{num_epochs}")
#     print(f"{'='*50}")
    
#     # Train
#     train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
#     print(f"Training Loss: {train_loss:.4f}")
    
#     # Validate
#     val_loss, val_accuracy, _, _ = evaluate(model, val_loader, device)
#     print(f"Validation Loss: {val_loss:.4f}")
#     print(f"Validation Accuracy: {val_accuracy:.4f}")
    
#     # Save best model
#     if val_accuracy > best_val_accuracy:
#         best_val_accuracy = val_accuracy
#         torch.save(model.state_dict(), 'best_distilbert_model.pt')
#         print("Best model saved!")
#         patience_counter = 0
#     else:
#         patience_counter += 1
#         if patience_counter >= patience:
#             print(f"Early stopping triggered after {epoch + 1} epochs")
#             break

# print("\nTraining completed!")

## Section 7: Evaluate on Validation and Test Sets

Evaluate model performance on both validation and test sets using accuracy, precision, recall, and F1-score metrics.

In [20]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_distilbert_model.pt'))

print("="*60)
print("FINAL MODEL EVALUATION")
print("="*60)

# Evaluate on validation set
print("\n--- Validation Set Evaluation ---")
val_loss, val_accuracy, val_predictions, val_true_labels = evaluate(model, val_loader, device)
val_precision = precision_score(val_true_labels, val_predictions, average='weighted')
val_recall = recall_score(val_true_labels, val_predictions, average='weighted')
val_f1 = f1_score(val_true_labels, val_predictions, average='weighted')

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation F1-Score: {val_f1:.4f}")

print("\nValidation Set Classification Report:")
print(classification_report(val_true_labels, val_predictions))

# Evaluate on test set
print("\n--- Test Set Evaluation ---")
test_loss, test_accuracy, test_predictions, test_true_labels = evaluate(model, test_loader, device)
test_precision = precision_score(test_true_labels, test_predictions, average='weighted')
test_recall = recall_score(test_true_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_true_labels, test_predictions, average='weighted')

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")

print("\nTest Set Classification Report:")
print(classification_report(test_true_labels, test_predictions))

# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Validation vs Test Performance")
print("="*60)
print(f"{'Metric':<15} {'Validation':<15} {'Test':<15}")
print("-"*60)
print(f"{'Accuracy':<15} {val_accuracy:<15.4f} {test_accuracy:<15.4f}")
print(f"{'Precision':<15} {val_precision:<15.4f} {test_precision:<15.4f}")
print(f"{'Recall':<15} {val_recall:<15.4f} {test_recall:<15.4f}")
print(f"{'F1-Score':<15} {val_f1:<15.4f} {test_f1:<15.4f}")

FINAL MODEL EVALUATION

--- Validation Set Evaluation ---


Evaluating: 100%|██████████| 615/615 [01:20<00:00,  7.59it/s]


Validation Loss: 0.6664
Validation Accuracy: 0.8913
Validation Precision: 0.9136
Validation Recall: 0.8913
Validation F1-Score: 0.8927

Validation Set Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.82      0.90      6000
           1       0.78      1.00      0.88      3838

    accuracy                           0.89      9838
   macro avg       0.89      0.91      0.89      9838
weighted avg       0.91      0.89      0.89      9838


--- Test Set Evaluation ---


Evaluating: 100%|██████████| 615/615 [01:21<00:00,  7.53it/s]

Test Loss: 0.6907
Test Accuracy: 0.8848
Test Precision: 0.9099
Test Recall: 0.8848
Test F1-Score: 0.8863

Test Set Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.81      0.90      6000
           1       0.77      1.00      0.87      3839

    accuracy                           0.88      9839
   macro avg       0.89      0.90      0.88      9839
weighted avg       0.91      0.88      0.89      9839


SUMMARY: Validation vs Test Performance
Metric          Validation      Test           
------------------------------------------------------------
Accuracy        0.8913          0.8848         
Precision       0.9136          0.9099         
Recall          0.8913          0.8848         
F1-Score        0.8927          0.8863         


In [21]:
# Tokenize the input text
input_text ="Having Grandma be your daycare can be awesome, but it can also suck.\
Can I just vent really quickly My MIL lives with us &amp; helps with child care. \
She basically functions as a live-in nanny.  \
She’s a sweet woman. But man, can she annoy me. She says things like “[toddler’s name] doesn’t do that when he’s with ME.” \
She sometimes calls him “my king,” I once heard her call him The King of Kings.  \
She calls my baby girl “my queen” &amp; also “mi reina” in Spanish. \
They’re too young for it to be a joke, it just feels a bit much to me. \
She constantly refers to them as “my babies” &amp; “my kids.”\
She constantly calls them to come to her to get them away from me &amp; says “Mommy’s busy” but I have to tell her it’s ok, they’re fine. \
She will keep trying to interact with them even if they’re interacting with me, but by talking about something completely different than what we are doing.  \
Again, she’s a sweet woman, &amp; we have some cultural differences, &amp; they are the most important things in her life, but sometimes I just want to be mommy without her around. \
And I feel selfish but I want to be more important to my kids than Grandma.My husband at least recognizes that she’s not perfect, \
she’s not the sharpest tool in the shed.  But he will do anything to not hurt her feelings &amp; heap tons of praise on her.\
But he’s comfortable being stern with me at times or lets me know he’s disappointed in something I’ve done but wouldn’t dare do that to her.\
I’ll stop there.  As you see I have conflicting feelings about my MIL.\
I’m grateful for how she helps us but sometimes want her to just leave us alone, even move out &amp; \
find other child care.  If anyone wants to suggest to post this to r/justnomil, I actually stay away from that sub so I don’t fuel my anger lol,\
but I apologize if you feel it’s inappropriate to post this here. \
Thank you so much if you’ve read this. \
It’s so hard to have an aging parent come live with you.  Can anyone relate?"
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=512)

# Move inputs to the same device as the model
inputs = {key: val.to(device) for key, val in inputs.items()}

# Perform inference
model.eval()
with torch.no_grad():
	outputs = model(**inputs)
	logits = outputs.logits
	predicted_class = torch.argmax(logits, dim=1).item()
print(logits)
print(f"Predicted class: {predicted_class}")

tensor([[ 2.9467, -3.0680]], device='cuda:0')
Predicted class: 0


****results summarty****
the model is outputing a great performence metrices on the current dataset but in real world domain it doesn't work that will

In [22]:


# Tokenize the input text
input_text ="Drinking alone I️ hate feeling this way. I’ve always been introverted but I’ve never felt this damn lonely."
input_text=minimal_preprocess(input_text)
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=512)

# Move inputs to the same device as the model
inputs = {key: val.to(device) for key, val in inputs.items()}

# Perform inference
model.eval()
with torch.no_grad():
	outputs = model(**inputs)
	logits = outputs.logits
	predicted_class = torch.argmax(logits, dim=1).item()
print(logits)
print(f"Predicted class: {predicted_class}")

tensor([[-4.0708,  5.2640]], device='cuda:0')
Predicted class: 1


In [23]:
input_text= "I [21/M] am unsure if I should tell my friends about my health issues. I am currently doing a couple of research projects at university in the biology building as a post-grad. My lab mates, all of whom are at least 3 years older than me, and I all care about each other as friends; we hang out, joke around, and like to keep a friendly atmosphere. This usually leads to most of them noticing when something is wrong with someone by the look on their face, and this is especially the case with me on some days. I had a blood test done recently which showed that 40% of my hemoglobin is HbS, which is characteristic of Sickle Cell trait and another co-existing condition, which is unknown to me at the moment (I'm working with my doctor to figure it out). Sometimes I show up to the lab in the morning and am just tired all around; it has no effect on my productivity, but I just don't feel like socializing at the moment, and some of my friends get the feeling that I'm trying to brush them off. In reality, I'm just exhausted, physically and mentally.\
What I'm worried about, because of past experiences with acquaintances, is that if I tell them about my chronic issue, it's going to cause a great deal of worry and/or stress over me, and I don't wish that at all, since I'm going to be working in the lab for the next year, and some of my lab mates have to present at conferences in a couple of months. I'm the type of person to work through tiredness and exhaustion if it means I can get some valuable output/results,\
so I'm not worried about myself. Am I analyzing a bit too much, or is this something my adviser and lab mates need to know?\
tl;dr: Chronic blood issue causes fatigue for me almost everyday, and friends and lab mates are worried about me looking tired all the time. Should I tell them, or keep saying that I'm alright?"

input_text=minimal_preprocess(input_text)
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=512)

# Move inputs to the same device as the model
inputs = {key: val.to(device) for key, val in inputs.items()}

# Perform inference
model.eval()
with torch.no_grad():
	outputs = model(**inputs)
	logits = outputs.logits
	predicted_class = torch.argmax(logits, dim=1).item()
print(logits)
print(f"Predicted class: {predicted_class}")


tensor([[-3.4400,  4.5856]], device='cuda:0')
Predicted class: 1


In [24]:
real_world_domain= pd.read_csv('non_mental_disorders_relevent_subreddits_control.csv')
real_world_domain=real_world_domain[20000:]
real_world_domain['label']=0
for idx , row in real_world_domain.iterrows():
    
    if pd.isna(row['other_posts']):
        real_world_domain.at[idx,'other_posts']=' '
real_world_domain['other_posts']=real_world_domain['other_posts'].apply(minimal_preprocess)
test_encodings = tokenize_function(real_world_domain['other_posts'])

real_world_domain_dataset = TextClassificationDataset(test_encodings, real_world_domain['label'])
real_world_domain_loader = DataLoader(real_world_domain_dataset, batch_size=batch_size, shuffle=False)




In [25]:

# Evaluate on test set
print("\n--- Test Set Evaluation ---")
test_loss, test_accuracy, test_predictions, test_true_labels = evaluate(model, real_world_domain_loader, device)
test_precision = precision_score(test_true_labels, test_predictions, average='weighted')
test_recall = recall_score(test_true_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_true_labels, test_predictions, average='weighted')

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")

print("\nTest Set Classification Report:")
print(classification_report(test_true_labels, test_predictions))

# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Validation vs Test Performance")
print("="*60)
print(f"{'Metric':<15} {'Validation':<15} {'Test':<15}")
print("-"*60)
print(f"{'Accuracy':<15} {val_accuracy:<15.4f} {test_accuracy:<15.4f}")
print(f"{'Precision':<15} {val_precision:<15.4f} {test_precision:<15.4f}")
print(f"{'Recall':<15} {val_recall:<15.4f} {test_recall:<15.4f}")
print(f"{'F1-Score':<15} {val_f1:<15.4f} {test_f1:<15.4f}")


--- Test Set Evaluation ---


Evaluating: 100%|██████████| 5032/5032 [11:02<00:00,  7.59it/s]


Test Loss: 3.1872
Test Accuracy: 0.4791
Test Precision: 1.0000
Test Recall: 0.4791
Test F1-Score: 0.6478

Test Set Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.48      0.65     80511
           1       0.00      0.00      0.00         0

    accuracy                           0.48     80511
   macro avg       0.50      0.24      0.32     80511
weighted avg       1.00      0.48      0.65     80511


SUMMARY: Validation vs Test Performance
Metric          Validation      Test           
------------------------------------------------------------
Accuracy        0.8913          0.4791         
Precision       0.9136          1.0000         
Recall          0.8913          0.4791         
F1-Score        0.8927          0.6478         


In [26]:
test_data=pd.read_csv('test_data_5000.csv')
test_data['label'] = (test_data['group'] == "distress").astype(int)
test_data=test_data[['response','label']]

test_data['response']=test_data['response'].apply(minimal_preprocess)
test_encodings=tokenizer(test_data['response'].tolist(),return_tensors='pt',padding=True,max_length=128)
test_dataset=TextClassificationDataset(test_encodings,test_data['label'])
test_dataloader=DataLoader(test_dataset,batch_size=16,shuffle=False)

In [27]:
test_data.head()

,response,label
0,"i'm feeling fairly balanced, staying productiv...",0
1,"i'm feeling fairly stable, staying consistent,...",0
2,"i'm feeling fairly okay, staying productive, a...",0
3,"i'm feeling fairly good, staying productive, a...",0
4,"i've been feeling overwhelmed and unfocused, m...",1


In [28]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_distilbert_model.pt'))

print("\n--- Test Set Evaluation ---")
test_loss, test_accuracy, test_predictions, test_true_labels = evaluate(model, test_dataloader, device)
test_precision = precision_score(test_true_labels, test_predictions, average='weighted')
test_recall = recall_score(test_true_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_true_labels, test_predictions, average='weighted')

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")

print("\nTest Set Classification Report:")
print(classification_report(test_true_labels, test_predictions))

# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Validation vs Test Performance")
print("="*60)
print(f"{'Metric':<15} {'Test':<15}")
print("-"*60)
print(f"{'Accuracy':<15}{test_accuracy:<15.4f}")
print(f"{'Precision':<15}{test_precision:<15.4f}")
print(f"{'Recall':<15}{test_recall:<15.4f}")
print(f"{'F1-Score':<15}{test_f1:<15.4f}")



--- Test Set Evaluation ---


Evaluating: 100%|██████████| 313/313 [00:03<00:00, 86.67it/s]

Test Loss: 4.4430
Test Accuracy: 0.5126
Test Precision: 0.2628
Test Recall: 0.5126
Test F1-Score: 0.3474

Test Set Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      2437
           1       0.51      1.00      0.68      2563

    accuracy                           0.51      5000
   macro avg       0.26      0.50      0.34      5000
weighted avg       0.26      0.51      0.35      5000


SUMMARY: Validation vs Test Performance
Metric          Test           
------------------------------------------------------------
Accuracy       0.5126         
Precision      0.2628         
Recall         0.5126         
F1-Score       0.3474         
